## Imports

In [2]:
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Bigdata") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()

print("Spark session started ✅")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 09:15:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started ✅


In [3]:
base_path = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
base_path = os.path.join(base_path, "data")

parquets = ["orders", "customers", "order_items", "payments", "products", "sellers", "reviews", "geolocation", "translation"]

dataframes = {}
i = 0
for parquet in parquets:
    df = spark.read.parquet(f"{os.path.join(base_path, "1_bronze", parquet)}")
    dataframes[parquet] = df
    i += 1
print(f"{i} dataframes successfully loaded ✅")

9 dataframes successfully loaded ✅


## Types

In [4]:
for name, df in dataframes.items():
    print(name.upper())
    df.printSchema()

ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

ORDER_ITEMS
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: doub

In [5]:
from pyspark.sql.types import DecimalType, ShortType

# Conversion of prices to DecimalType for better precision and conversion of small integers to ShortType

dataframes["order_items"] = dataframes["order_items"] \
    .withColumn("order_item_id", dataframes["order_items"]["order_item_id"].cast(ShortType())) \
    .withColumn("price", dataframes["order_items"]["price"].cast(DecimalType(10, 2))) \
    .withColumn("freight_value", dataframes["order_items"]["freight_value"].cast(DecimalType(10, 2)))


dataframes["payments"] = dataframes["payments"] \
    .withColumn("payment_sequential", dataframes["payments"]["payment_sequential"].cast(ShortType())) \
    .withColumn("payment_installments", dataframes["payments"]["payment_installments"].cast(ShortType())) \
    .withColumn("payment_value", dataframes["payments"]["payment_value"].cast(DecimalType(10, 2)))


dataframes["products"] = dataframes["products"] \
    .withColumn("product_name_lenght", dataframes["products"]["product_name_lenght"].cast(ShortType())) \
    .withColumn("product_description_lenght", dataframes["products"]["product_description_lenght"].cast(ShortType())) \
    .withColumn("product_photos_qty", dataframes["products"]["product_photos_qty"].cast(ShortType())) \
    .withColumn("product_length_cm", dataframes["products"]["product_length_cm"].cast(ShortType())) \
    .withColumn("product_height_cm", dataframes["products"]["product_height_cm"].cast(ShortType())) \
    .withColumn("product_width_cm", dataframes["products"]["product_width_cm"].cast(ShortType()))


dataframes["reviews"] = dataframes["reviews"].withColumn("review_score", dataframes["reviews"]["review_score"].cast(ShortType()))


## Null values

In [6]:
for name, df in dataframes.items():
    print(f"\n{name.upper()} :")
    for col in df.columns:
        null_count = df.filter(df[col].isNull()).count()
        if null_count > 0:
            print(f"  {col} : {null_count} nulls")

print("For the reviews dataframe, the null values are only reviews without comment, only a grade.")


ORDERS :
  order_approved_at : 160 nulls
  order_delivered_carrier_date : 1783 nulls
  order_delivered_customer_date : 2965 nulls

CUSTOMERS :

ORDER_ITEMS :

PAYMENTS :

PRODUCTS :
  product_category_name : 610 nulls
  product_name_lenght : 610 nulls
  product_description_lenght : 610 nulls
  product_photos_qty : 610 nulls
  product_weight_g : 2 nulls
  product_length_cm : 2 nulls
  product_height_cm : 2 nulls
  product_width_cm : 2 nulls

SELLERS :

REVIEWS :
  review_comment_title : 87656 nulls
  review_comment_message : 58247 nulls

GEOLOCATION :

TRANSLATION :
For the reviews dataframe, the null values are only reviews without comment, only a grade.


### Orders

In [7]:
from pyspark.sql import functions as F

print("Null values in the orders dataframe, by order status:")

dataframes["orders"].groupBy("order_status").agg(
    F.sum(F.when(F.col("order_approved_at").isNull(), 1).otherwise(0)).alias("null_approved"),
    F.sum(F.when(F.col("order_delivered_carrier_date").isNull(), 1).otherwise(0)).alias("null_carrier"),
    F.sum(F.when(F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("null_customer")
).orderBy("order_status").show()

delivered_not_approved = dataframes["orders"].filter((F.col("order_status") == "delivered") & (F.col("order_approved_at").isNull()))
print(f"The {delivered_not_approved.count()} delivered orders without approval date are illogical, we replace them with the average approval delay.")


df_orders_approved = dataframes["orders"].filter(F.col("order_approved_at").isNotNull()) \
    .select(F.avg(F.unix_timestamp("order_approved_at") - F.unix_timestamp("order_purchase_timestamp")).alias("avg_approval_delay"))
avg_approval_delay = round(df_orders_approved.collect()[0]["avg_approval_delay"])


dataframes["orders"] = dataframes["orders"].withColumn(
    "order_approved_at", 
    F.when(
        F.col("order_approved_at").isNull() & (F.col("order_status") == "delivered"), 
        F.col("order_purchase_timestamp") + F.expr(f"INTERVAL {int(avg_approval_delay)} SECOND")
    ).otherwise(F.col("order_approved_at"))
)

print("We apply the same logic for the orders where the approval time is before the purchase time.")
dataframes["orders"] = dataframes["orders"].withColumn(
    "order_approved_at", 
    F.when(
        F.col("order_approved_at") < F.col("order_purchase_timestamp"), 
        F.col("order_purchase_timestamp") + F.expr(f"INTERVAL {int(avg_approval_delay)} SECOND")
    ).otherwise(F.col("order_approved_at"))
)


Null values in the orders dataframe, by order status:
+------------+-------------+------------+-------------+
|order_status|null_approved|null_carrier|null_customer|
+------------+-------------+------------+-------------+
|    approved|            0|           2|            2|
|    canceled|          141|         550|          619|
|     created|            5|           5|            5|
|   delivered|           14|           2|            8|
|    invoiced|            0|         314|          314|
|  processing|            0|         301|          301|
|     shipped|            0|           0|         1107|
| unavailable|            0|         609|          609|
+------------+-------------+------------+-------------+

The 14 delivered orders without approval date are illogical, we replace them with the average approval delay.
We apply the same logic for the orders where the approval time is before the purchase time.


## Inconsistencies

In [8]:
print("Null or negative payments:")
dataframes["payments"].filter(F.col("payment_value") == 0).show()

print("These payments are vouchers or probably reductions so we don't need to handle them.")
print("However, we replace the not_defined payment types with voucher.")

dataframes["payments"] = dataframes["payments"].withColumn(
    "payment_type", 
    F.when(F.col("payment_type") == "not_defined", "voucher").otherwise(F.col("payment_type"))
)

Null or negative payments:
+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|8bcbe01d44d147f90...|                 4|     voucher|                   1|         0.00|
|fa65dad1b0e818e3c...|                14|     voucher|                   1|         0.00|
|6ccb433e00daae128...|                 4|     voucher|                   1|         0.00|
|4637ca194b6387e2d...|                 1| not_defined|                   1|         0.00|
|00b1cb0320190ca0d...|                 1| not_defined|                   1|         0.00|
|45ed6e85398a87c25...|                 3|     voucher|                   1|         0.00|
|fa65dad1b0e818e3c...|                13|     voucher|                   1|         0.00|
|c8c528189310eaa44...|                 1| not_defined|                   

In [9]:
print("Other inconsistencies for the orders dataframe:")
print(f"   Delivered orders without customer delivery date: {dataframes["orders"].filter((F.col('order_status') == 'delivered') & F.col('order_delivered_customer_date').isNull()).count()}")
print(f"   Delivered orders without carrier date : {dataframes["orders"].filter((F.col('order_status') == 'delivered') & F.col('order_delivered_carrier_date').isNull()).count()}")


Other inconsistencies for the orders dataframe:
   Delivered orders without customer delivery date: 8
   Delivered orders without carrier date : 2


## Duplicates

In [10]:
for name, df in dataframes.items():
    print(f"{name.upper()} : {df.count() - df.distinct().count()} duplicates")

ORDERS : 0 duplicates
CUSTOMERS : 0 duplicates
ORDER_ITEMS : 0 duplicates
PAYMENTS : 0 duplicates
PRODUCTS : 0 duplicates
SELLERS : 0 duplicates
REVIEWS : 0 duplicates


GEOLOCATION : 261831 duplicates
TRANSLATION : 0 duplicates


For the geolocation dataframe, we need to keep only distinct zip_codes since it is a join key with two other dataframes.  
There are several latitude / longitude combos for a single zip_code, so we use the average latitude and average latitude for each zip_code.

In [11]:
from pyspark.sql import functions as F

geo_duplicates = dataframes["geolocation"].groupBy("geolocation_zip_code_prefix").count().filter("count > 1")
print("GEOLOCATION - Duplicated zip codes")
geo_duplicates.show(5)


df_geo_unique = dataframes["geolocation"].groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng"),
    F.first("geolocation_city").alias("geolocation_city"),
    F.first("geolocation_state").alias("geolocation_state")
)

print("\n\nGEOLOCATION before removing duplicates -", dataframes["geolocation"].count(), "lines")
dataframes["geolocation"] = df_geo_unique
print("GEOLOCATION after removing duplicates -", dataframes["geolocation"].count(), "lines")
dataframes["geolocation"].show(5)


GEOLOCATION - Duplicated zip codes
+---------------------------+-----+
|geolocation_zip_code_prefix|count|
+---------------------------+-----+
|                       1238|  164|
|                       2122|   33|
|                       2142|    5|
|                       2366|   33|
|                       2866|   41|
+---------------------------+-----+
only showing top 5 rows


GEOLOCATION before removing duplicates - 1000163 lines
GEOLOCATION after removing duplicates - 19015 lines
+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                       1001|-23.550189776551765|-46.634023555904214|       sao paulo|               SP|
|                       1002| -23.54814573176355| -46.63497921074498|       sao paulo|  

## Basic joins

In [12]:
df_products = dataframes["products"]
df_products_joined = df_products.join(dataframes["translation"], on="product_category_name", how="left")
dataframes["products"] = df_products_joined
del dataframes["translation"]


df_geo = dataframes["geolocation"]
df_geo = df_geo.drop("geolocation_city", "geolocation_state")

df_customers = dataframes["customers"]
df_customers_joined = df_customers.join(df_geo, on=df_customers["customer_zip_code_prefix"] == df_geo["geolocation_zip_code_prefix"], how="left")
df_customers_joined = df_customers_joined.drop("geolocation_zip_code_prefix")
dataframes["customers"] = df_customers_joined

df_sellers = dataframes["sellers"]
df_sellers_joined = df_sellers.join(df_geo, on=df_sellers["seller_zip_code_prefix"] == df_geo["geolocation_zip_code_prefix"], how="left")
df_sellers_joined = df_sellers_joined.drop("geolocation_zip_code_prefix")
dataframes["sellers"] = df_sellers_joined

del dataframes["geolocation"]

## Exports

In [13]:
for name, df in dataframes.items():
    df.write.mode("overwrite").parquet(f"{base_path}/2_silver/{name}")
    print(f"Dataframe {name} saved to silver ✅")

Dataframe orders saved to silver ✅
Dataframe customers saved to silver ✅
Dataframe order_items saved to silver ✅
Dataframe payments saved to silver ✅
Dataframe products saved to silver ✅
Dataframe sellers saved to silver ✅
Dataframe reviews saved to silver ✅
